# 22_GameSkeleton_AB — Thử luật chơi mới: đi từ ô A tới ô B

Model: `step2 s1` (`Data_321/checkpoints_task9/step2/seed1/TCN.pt`). Chỉ chạy model, **không train**. `20_`, `21_` và `game_ui` giữ nguyên, chưa đụng. Cell 1 đến 4 và cell 8 chép nguyên từ 21_ (đã test ổn); cell 5, 6, 7, 9 là phần mới.

**Cách chạy:** kích hoạt `C:\\envs\\train-gpu\\Scripts\\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 9. Cell 6 tự kiểm tra luật bằng dữ liệu giả, cell 9 mở webcam.

**Phím:** `SPACE` chơi / chơi tiếp / chơi lại | `r` reset điểm | `q` thoát.

**Luật A → B:**
1. SPACE rồi đưa tay vào ô A. Tay chạm A thì bắt đầu, không đếm gì cả.
2. Đi theo đường nối tới ô B. Lệch ít thì cho qua; ra khỏi dải thì đường đỏ lên; đỏ quá `off_max` giây hoặc mất tay quá `lost_max` giây thì về lại chờ ở A.
3. Tay chạm B: model chốt đáp án từ khung đó (chỉ dùng thông tin tới lúc đó) rồi khóa, AI ra quân khắc chế.
4. Đợi `confirm_sec` = 0,2 giây, đọc cử chỉ thật bằng luật ngón tay (độc lập với model). Giống lúc chốt thì ra kết quả. Khác thì **hủy ván**, không tính điểm, không hiện đáp án AI, hiện khung lúc chạm B, SPACE để chơi lại. Mất tay suốt 0,2 giây thì "không xác định", không tính điểm.

**Tham số** (đổi ở cell 9, dòng `g = GameAB(...)`): `A`, `B` (toạ độ 0-1 trên màn hình đã lật gương), `box_r` (bán kính ô A, B), `band` (độ lệch cho phép), `off_max`, `lost_max`, `confirm_sec`. `HAND_MODE`: `palm` (điểm giữa lòng bàn tay) hoặc `box` (tâm ô bao quanh tay).


In [1]:
#cell 1: import, đường dẫn, kiểm tra file cần có, bật CUDA
import time, math, collections
import numpy as np, torch, cv2
import torch.nn as nn, torch.nn.functional as F
import mediapipe as mp
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không chạy trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
TASK = BASE / "hand_landmarker.task"
CKPT = BASE / "checkpoints_task9" / "step2" / "seed1" / "TCN.pt"      # step2 s1
for p in (TASK, CKPT):
    assert p.exists(), f"Không thấy: {p}"
CLASSES = ["paper", "rock", "scissors"]                              # nhãn 0, 1, 2
print("OK | GPU:", torch.cuda.get_device_name(0))

OK | GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
#cell 2: TCN + 546 đặc trưng (giống 13/19), nạp checkpoint step2 s1, làm nóng GPU
class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__(); self.pad = 2 * d
        self.conv = nn.Conv1d(c, c, 3, dilation=d); self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, in_features=126, c=64):
        super().__init__(); self.inp = nn.Linear(in_features, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))

PAIR_I, PAIR_J = torch.triu_indices(21, 21, offset=1, device=dev)    # 210 cặp khớp

def features_from_pos(pos, hand_mask):
    """pos (b,T,21,3) -> (b,T,546): 126 gốc + 210 khoảng cách + 210 vận tốc khoảng cách (giống 13)."""
    pos = torch.where(hand_mask[:, :, None, None], pos, torch.zeros_like(pos))
    b, T = pos.shape[:2]
    flat = pos.reshape(b, T, 63)
    pv = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    vel = torch.zeros_like(flat); vel[:, 1:] = (flat[:, 1:] - flat[:, :-1]) * pv
    dist = torch.linalg.vector_norm(pos[:, :, PAIR_I] - pos[:, :, PAIR_J], dim=-1) * hand_mask[:, :, None]
    dvel = torch.zeros_like(dist); dvel[:, 1:] = (dist[:, 1:] - dist[:, :-1]) * pv
    return torch.cat([flat, vel, dist, dvel], dim=-1)

def geo_from_base(X, hand_mask):
    return features_from_pos(X[..., :63].reshape(X.shape[0], X.shape[1], 21, 3), hand_mask)

net = TCN(in_features=546).to(dev)
ck = torch.load(CKPT, map_location=dev, weights_only=False)
assert ck.get("cfg", {}).get("n_in") == 546, f"Checkpoint không phải họ step2: {ck.get('cfg')}"
net.load_state_dict(ck["w"]); net.eval()
with torch.no_grad(): net(torch.zeros(1, 128, 546, device=dev))
print(f"Đã nạp step2 s1 | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")

Đã nạp step2 s1 | epoch 27 | val_loss 0.3224


In [3]:
#cell 3: MediaPipe (CPU), nội suy webcam -> 120 fps, hàm trả xác suất 3 lớp của khung cuối (giống 13/19)
landmarker = mp.tasks.vision.HandLandmarker.create_from_options(mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO, num_hands=1))
T0 = time.perf_counter()          # timestamp MediaPipe phải luôn tăng, kể cả khi chạy lại cell 10

def detect(frame_bgr, ts_ms):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = landmarker.detect_for_video(mp.Image(mp.ImageFormat.SRGB, rgb), ts_ms)
    if not res.hand_landmarks:
        return None
    return np.array([[p.x, p.y, p.z] for p in res.hand_landmarks[0]], np.float32)

FPS_MODEL, WIN = 120, 128
buf = collections.deque(maxlen=WIN)
state = {"prev_lm": None, "prev_t": None}

def push(lm, t):
    prev, pt = state["prev_lm"], state["prev_t"]
    n = 1 if pt is None else max(1, round((t - pt) * FPS_MODEL))
    if lm is None:
        buf.extend([np.full((21, 3), np.nan, np.float32)] * n)
    elif prev is None:
        buf.append(lm)
    else:
        buf.extend(prev + (lm - prev) * (k / n) for k in range(1, n + 1))
    state["prev_lm"], state["prev_t"] = lm, t

def base_features(lm):
    has = ~np.isnan(lm[:, 0, 0])
    q = lm - lm[:, :1]
    q = np.nan_to_num(q / np.maximum(np.linalg.norm(q[:, 9], axis=1)[:, None, None], 1e-8)).astype(np.float32)
    q[~has] = 0
    pos = q.reshape(len(q), 63)
    vel = np.zeros_like(pos)
    if len(pos) > 1:
        vel[1:] = (pos[1:] - pos[:-1]) * (has[1:] & has[:-1])[:, None]
    return np.hstack([pos, vel]).astype(np.float32), has

def probs_now():
    """Xác suất (paper, rock, scissors) của khung cuối; None nếu khung cuối mất tay."""
    lm = np.stack(buf)
    if np.isnan(lm[-1, 0, 0]):
        return None
    base, has = base_features(lm)
    xb, mb = torch.from_numpy(base)[None].to(dev), torch.from_numpy(has)[None].to(dev)
    with torch.no_grad():
        return net(geo_from_base(xb, mb)).softmax(-1)[0, -1].cpu().numpy()
print("Sẵn sàng: MediaPipe + nội suy 120 fps + hàm probs_now()")

Sẵn sàng: MediaPipe + nội suy 120 fps + hàm probs_now()


In [4]:
#cell 4: luật đọc hình tay ngay trên 1 khung (độc lập với TCN), chấm điểm mềm nên khung nào có tay cũng ra đáp án
TIP, PIP = [8, 12, 16, 20], [6, 10, 14, 18]          # trỏ, giữa, áp út, út (bỏ ngón cái)
SIGN = np.array([[1, 1, 1, 1], [-1, -1, -1, -1], [1, 1, -1, -1]], np.float32)   # mẫu ngón duỗi(+)/gập(-): paper, rock, scissors

def finger_scores(lm):
    """lm (...,21,3) -> (...,4): log(đầu ngón / khớp giữa ngón) tính từ cổ tay; > 0 là duỗi, < 0 là gập, càng xa 0 càng chắc."""
    d = np.linalg.norm(lm - lm[..., :1, :], axis=-1)
    return np.log((d[..., TIP] + 1e-6) / (d[..., PIP] + 1e-6))

def hand_shape(lm, margin=0.0):
    """0 paper | 1 rock | 2 scissors; -1 chỉ khi mất tay (hoặc 2 lớp đứng đầu cách nhau < margin, mặc định 0 = luôn chọn một lớp)."""
    score = finger_scores(lm) @ SIGN.T                                    # (...,3) điểm khớp với từng mẫu
    top = np.sort(score, axis=-1)
    out = np.where(top[..., -1] - top[..., -2] < margin, -1, score.argmax(-1))
    return np.where(np.isnan(lm).any(axis=(-1, -2)), -1, out)
print("Đã định nghĩa hand_shape.")

Đã định nghĩa hand_shape.


In [5]:
#cell 5: luật game A -> B (búa ở A giữ hold_a giây; giữ búa tới khi chạm vạch gate_f rồi mới được đổi tay); không dính webcam nên test được riêng ở cell 6
COUNTER = {0: 2, 1: 0, 2: 1}      # người ra paper -> AI ra scissors; rock -> paper; scissors -> rock

def hand_point(lm, mirror=True, mode="palm"):
    """Điểm đại diện của bàn tay trên màn hình (toạ độ 0-1, lật gương nếu mirror).
    palm: trung bình cổ tay + 4 khớp gốc ngón (ít đổi khi xoè/nắm tay) | box: tâm ô bao quanh tay."""
    xy = lm[:, :2]
    p = (xy.min(0) + xy.max(0)) / 2 if mode == "box" else xy[[0, 5, 9, 13, 17]].mean(0)
    return np.array([1 - p[0] if mirror else p[0], p[1]], np.float32)

class GameAB:
    def __init__(self, A=(0.2, 0.5), B=(0.8, 0.5), ar=4 / 3, box_r=0.13, band=0.09,
                 off_max=0.5, lost_max=0.3, K=1, confirm_sec=0.2, thr=0.6, hold_a=0.3, gate_f=0.25, fist_grace=0.1):
        self.A_n, self.B_n, self.ar = A, B, ar                       # A, B: toạ độ 0-1 trên màn hình (để vẽ)
        self.A, self.B = np.array([A[0] * ar, A[1]]), np.array([B[0] * ar, B[1]])   # để tính khoảng cách: đơn vị = chiều cao khung hình
        assert gate_f > box_r / np.linalg.norm(self.B - self.A), "Vạch phải nằm ngoài ô A (tăng gate_f hoặc giảm box_r)"
        self.box_r, self.band, self.off_max, self.lost_max = box_r, band, off_max, lost_max
        self.K, self.confirm_sec, self.thr, self.hold_a = K, confirm_sec, thr, hold_a
        self.gate_f, self.fist_grace = gate_f, fist_grace            # gate_f: vị trí vạch trên đường A->B (0 là A, 1 là B) | fist_grace: sai hình tối đa (giây) trước vạch
        self.score = {"ai": 0, "player": 0, "draw": 0}
        self.stats = {"ok": 0, "cancel": 0, "unknown": 0}            # đếm ván: bình thường / huỷ / không xác định
        self.history = []                                            # nhật ký mỗi ván (mốc chốt, đáp án, kết quả)
        self.state, self.msg, self.base_msg = "WAIT", "Nhan SPACE de choi", ""
        self.off = self.locked_now = self.entered_go = False         # off: tay lệch đường (đường đỏ) | locked_now: đúng khung chốt | entered_go: đúng khung bắt đầu đi
        self.hold_since, self.a_hold = None, 0.0                     # a_hold: tiến độ giữ búa ở ô A, 0 đến 1
        self.passed = self.warn = False                              # passed: tay đã chạm vạch | warn: đang sai hình trước vạch (cảnh báo)
        self.bad_since = None
        self.guess = self.ai = self.player = self.outcome = None

    def start(self, t=None):
        """Bấm SPACE: chơi ván đầu, chơi tiếp sau kết quả, hoặc chơi lại sau khi huỷ. Đang giữa ván thì bỏ qua."""
        if self.state not in ("WAIT", "RESULT", "CANCEL"):
            return
        self.guess = self.ai = self.player = self.outcome = None
        self.off, self.hold_since, self.a_hold = False, None, 0.0
        self.passed = self.warn = False; self.bad_since = None
        self.state = "NEED_A"
        self.msg = self.base_msg = "Nam tay thanh BUA, dua vao o A"

    def _xy(self, pt):
        return np.array([pt[0] * self.ar, pt[1]])

    def _seg_dist(self, p):
        ab = self.B - self.A
        f = np.clip(np.dot(p - self.A, ab) / np.dot(ab, ab), 0, 1)
        return float(np.linalg.norm(p - (self.A + f * ab)))

    def _prog(self, p):
        """Tay đã đi được bao nhiêu phần đường A->B (0 tại A, 1 tại B)."""
        ab = self.B - self.A
        return float(np.dot(p - self.A, ab) / np.dot(ab, ab))

    def _abort(self, msg):
        self.state, self.off, self.hold_since, self.a_hold = "NEED_A", False, None, 0.0
        self.passed = self.warn = False; self.bad_since = None
        self.msg = self.base_msg = msg

    def step(self, t, pt, probs, shape):
        """Gọi mỗi khung webcam. pt: điểm tay (None nếu mất tay); probs: 3 xác suất (None nếu mất tay); shape: hình tay đọc bằng luật ngón tay."""
        self.locked_now = self.entered_go = False
        has = pt is not None and probs is not None
        p = self._xy(pt) if has else None
        if self.state == "NEED_A":
            in_a = has and np.linalg.norm(p - self.A) <= self.box_r
            if in_a and shape == 1:                                  # búa trong ô A: bắt đầu tính giờ giữ
                if self.hold_since is None:
                    self.hold_since = t
                self.a_hold = min(1.0, (t - self.hold_since) / self.hold_a)
                self.msg = "Giu BUA trong o A"
                if self.a_hold >= 1.0 - 1e-9:                        # giữ đủ -> bắt đầu đi
                    self.state, self.msg, self.t_go, self.entered_go = "GO", "", t, True
                    self.lost_since = self.off_since = self.bad_since = None
                    self.passed = self.warn = False
                    self.hist = collections.deque([probs], maxlen=self.K)
            else:                                                    # sai hình, ra khỏi ô A hoặc mất tay: giữ lại từ đầu
                self.hold_since, self.a_hold = None, 0.0
                self.msg = "Nam tay thanh BUA (o A)" if in_a else self.base_msg
        elif self.state == "GO":
            self._go(t, p, probs, has, shape)
        elif self.state == "CONFIRM":
            self._confirm(t, shape)

    def _go(self, t, p, probs, has, shape):
        if not has:                                                  # mất tay quá lost_max -> về chờ ở A
            self.lost_since = self.lost_since if self.lost_since is not None else t
            if t - self.lost_since > self.lost_max:
                self._abort("Mat tay! Dua BUA ve o A")
            return
        self.lost_since = None
        self.hist.append(probs)
        if not self.passed:                                          # chưa chạm vạch: tay phải còn là búa
            if self._prog(p) >= self.gate_f:
                self.passed, self.warn, self.bad_since = True, False, None     # chạm vạch (chỉ cần điểm tay chạm) -> được đổi tay
            elif shape == 1:
                self.warn, self.bad_since = False, None
            else:
                self.warn = True                                     # cảnh báo; sai hình quá fist_grace giây thì về A
                self.bad_since = self.bad_since if self.bad_since is not None else t
                if t - self.bad_since > self.fist_grace:
                    self._abort("Canh bao: phai giu BUA toi vach! Dua BUA ve o A")
                    return
        near_a, near_b = np.linalg.norm(p - self.A) <= self.box_r, np.linalg.norm(p - self.B) <= self.box_r
        self.off = not (near_a or near_b or self._seg_dist(p) <= self.band)
        if self.off:                                                 # lệch đường: đỏ lên, đỏ quá off_max thì về chờ ở A
            self.off_since = self.off_since if self.off_since is not None else t
            if t - self.off_since > self.off_max:
                self._abort("Lech duong! Dua BUA ve o A")
                return
        else:
            self.off_since = None
        if near_b:                                                   # chạm B: chốt đáp án ngay, không random, không sửa sau đó
            pm = np.mean(self.hist, axis=0)
            self.guess = int(pm.argmax()); self.ai = COUNTER[self.guess]
            self.t_lock, self.win = t, []
            self.state, self.locked_now, self.off = "CONFIRM", True, False

    def _confirm(self, t, shape):
        if shape is not None and shape >= 0:                         # khung chốt không tính; bỏ qua khung mất tay
            self.win.append(int(shape))
        if t - self.t_lock < self.confirm_sec:
            return
        if not self.win:                                             # mất tay suốt cửa sổ -> không xác định, không tính điểm
            self.outcome, self.state = "unknown", "RESULT"; self.stats["unknown"] += 1
        else:
            cnt = np.bincount(self.win, minlength=3)
            top = [c for c in range(3) if cnt[c] == cnt.max()]
            self.player = next(c for c in reversed(self.win) if c in top)   # hoà phiếu: lấy cử chỉ muộn nhất
            if self.player != self.guess:                            # khác lúc chốt -> huỷ ván, không tính điểm
                self.state, self.outcome = "CANCEL", None; self.stats["cancel"] += 1
            else:
                d = (self.ai - self.player) % 3
                self.outcome = ("draw", "player", "ai")[d]
                self.score[self.outcome] += 1; self.state = "RESULT"; self.stats["ok"] += 1
        self.history.append({"tu_A_den_chot_s": round(self.t_lock - self.t_go, 2), "model_doan": self.guess,
                             "tay_sau_xac_nhan": self.player, "khung_xac_nhan": list(self.win),
                             "ket_qua": self.state, "outcome": self.outcome})
print("Đã định nghĩa GameAB (búa ở A + vạch).")

Đã định nghĩa GameAB (búa ở A + vạch).


In [6]:
#cell 6: tự kiểm tra luật A -> B (búa ở A + vạch) bằng dữ liệu giả, không cần webcam
rock, paper, dt = np.array([.05, .90, .05]), np.array([.90, .05, .05]), 1 / 30
A_, B_ = np.array([0.2, 0.5]), np.array([0.8, 0.5])      # mặc định của GameAB

def go(g, pt, probs=rock, shape=1, n=1):
    for _ in range(n):
        go.t += dt; g.step(go.t, pt, probs, shape); go.locks += g.locked_now

def along(f): return A_ + (B_ - A_) * f                     # f=0 là ô A, f=1 là ô B

def start_go(g, probs=rock):                                # SPACE rồi nắm búa giữ ở A đủ hold_a giây -> GO
    g.start(); go(g, along(0), probs, 1, n=12); assert g.state == "GO", g.state

def walk(g, shape=1, probs=rock, n=20):                     # (nếu còn ở NEED_A thì giữ búa ở A trước) rồi đi A -> B: búa tới vạch, sau vạch ra `shape`
    if g.state == "NEED_A": go(g, along(0), probs, 1, n=12)
    for i in range(n + 1):
        go(g, along(i / n), probs, 1 if i / n < g.gate_f else shape)
        if g.state != "GO": break

# 1) đi đúng đường: chưa SPACE thì không làm gì; búa giữ ở A đủ giờ mới chạy; chạm B chốt đúng 1 lần; sau 0.2 s ra kết quả
go.t, go.locks = 0.0, 0
g = GameAB(); assert g.state == "WAIT"
go(g, along(0)); assert g.state == "WAIT"
g.start(); assert g.state == "NEED_A"
go(g, along(0.5)); assert g.state == "NEED_A"                # tay chưa vào ô A thì chưa chạy
walk(g); assert g.state == "CONFIRM" and g.guess == 1 and g.ai == 0 and g.locked_now
go(g, along(1), n=9)
assert g.state == "RESULT" and g.outcome == "ai" and g.score["ai"] == 1 and g.stats["ok"] == 1 and go.locks == 1
assert len(g.history) == 1 and g.history[0]["ket_qua"] == "RESULT"

# 2) búa nằm yên ở A: vào GO rồi không huỷ; SPACE giữa ván bị bỏ qua
go.t = 0.0; g = GameAB(); g.start(); go(g, along(0), n=60)
assert g.state == "GO"; g.start(); assert g.state == "GO"

# 3) lệch ngắn: đỏ nhưng vẫn chạy; về lại đường thì hết đỏ; lệch quá off_max thì về chờ ở A
go.t = 0.0; g = GameAB(); start_go(g)
go(g, along(0.5) + [0, 0.15], n=6); assert g.state == "GO" and g.off
go(g, along(0.5)); assert g.state == "GO" and not g.off
go(g, along(0.5) + [0, 0.15], n=20); assert g.state == "NEED_A" and "Lech" in g.msg

# 4) tay lệch nhẹ nhưng còn trong ô A thì không đỏ
go.t = 0.0; g = GameAB(); g.start(); go(g, along(0) + [0.05, 0], n=12); assert g.state == "GO" and not g.off

# 5) mất tay ngắn (0.2 s) vẫn chạy; mất lâu thì về chờ ở A; búa giữ lại ở A đủ giờ thì chạy lại
go.t = 0.0; g = GameAB(); start_go(g)
go(g, None, None, None, n=6); assert g.state == "GO"
go(g, None, None, None, n=10); assert g.state == "NEED_A"
go(g, along(0), n=12); assert g.state == "GO"

# 6) AI chốt tại B rồi không đổi: người chơi đổi sang paper sau đó -> huỷ ván, không tính điểm; SPACE -> chơi lại
go.t = 0.0; g = GameAB(); g.start(); walk(g, shape=1)
assert g.state == "CONFIRM" and g.guess == 1 and g.ai == 0
go(g, along(1), probs=paper, shape=0, n=9)
assert g.state == "CANCEL" and g.guess == 1 and g.ai == 0 and g.outcome is None
assert sum(g.score.values()) == 0 and g.stats["cancel"] == 1
g.start(); assert g.state == "NEED_A" and g.outcome is None and g.guess is None

# 7) mất tay suốt cửa sổ xác nhận -> không xác định, không tính điểm
go.t = 0.0; g = GameAB(); g.start(); walk(g); go(g, None, None, None, n=9)
assert g.state == "RESULT" and g.outcome == "unknown" and sum(g.score.values()) == 0 and g.stats["unknown"] == 1

# 8) hoà phiếu trong cửa sổ lấy cử chỉ muộn nhất (AI chốt paper)
for seq, want in (([1, 1, 0, 0], "RESULT"), ([0, 0, 1, 1], "CANCEL")):
    go.t = 0.0; g = GameAB(confirm_sec=0.12); g.start(); walk(g, probs=paper)
    for s in seq: go(g, along(1), probs=paper, shape=s)
    assert g.state == want, (seq, g.state)

# 9) hand_point: lật gương đúng
lm = np.zeros((21, 3), np.float32); lm[:, 0], lm[:, 1] = 0.2, 0.6
assert np.allclose(hand_point(lm, True), [0.8, 0.6]) and np.allclose(hand_point(lm, False), [0.2, 0.6])

# 10) búa ở A giữ đủ hold_a (0.3 s)
for sh in (0, 2):                                            # paper hoặc scissors ở A: không bao giờ bắt đầu được
    go.t = 0.0; g = GameAB(); g.start(); go(g, along(0), shape=sh, n=60)
    assert g.state == "NEED_A" and g.a_hold == 0 and "BUA" in g.msg, (sh, g.state, g.msg)
go.t = 0.0; g = GameAB(); g.start(); go(g, along(0), shape=1, n=5)       # búa chưa giữ đủ: tiến độ giữa 0 và 1, chưa đi
assert g.state == "NEED_A" and 0 < g.a_hold < 1
go(g, along(0), shape=1, n=7); assert g.state == "GO"
go.t = 0.0; g = GameAB(); g.start(); go(g, along(0), shape=1, n=6)       # đang giữ mà mở tay (paper) -> tiến độ về 0, phải giữ lại từ đầu
assert g.a_hold > 0
go(g, along(0), probs=paper, shape=0); assert g.a_hold == 0
go(g, along(0), shape=1, n=6); assert g.state == "NEED_A"
go(g, along(0), shape=1, n=6); assert g.state == "GO"
go.t = 0.0; g = GameAB(); g.start(); go(g, along(0), shape=1, n=6)       # mất tay khi đang giữ -> về 0
go(g, None, None, None); assert g.a_hold == 0 and g.state == "NEED_A"
go.t = 0.0; g = GameAB(); g.start(); go(g, along(0.5), shape=1, n=30)    # búa nhưng ngoài ô A: không tính
assert g.state == "NEED_A" and g.a_hold == 0

# 11) VẠCH: đã vào GO thì còn phải giữ búa tới khi chạm vạch (gate_f = 0.25); chạm vạch rồi đổi gì cũng được
go.t = 0.0; g = GameAB(); start_go(g); walk(g, shape=0, probs=paper)    # búa tới vạch, sau vạch ra paper: bình thường
assert g.state == "CONFIRM" and g.guess == 0
go.t = 0.0; g = GameAB(); start_go(g)                                   # đổi tay ngay ở A (chưa chạm vạch)
go(g, along(0), probs=paper, shape=0, n=2); assert g.state == "GO" and g.warn          # mới 2 khung: cảnh báo, chưa huỷ
go(g, along(0), probs=paper, shape=0, n=4); assert g.state == "NEED_A" and "BUA" in g.msg and not g.passed
go(g, along(0), n=12); assert g.state == "GO"                           # giữ búa lại ở A đủ giờ thì đi lại được
go.t = 0.0; g = GameAB(); start_go(g)                                   # nhấp nháy 1 khung sai hình rồi lại búa: bỏ qua
go(g, along(0.1), probs=paper, shape=0); assert g.warn and g.state == "GO"
go(g, along(0.1)); assert g.state == "GO" and not g.warn
go.t = 0.0; g = GameAB(); start_go(g)                                   # đổi tay khi sắp tới vạch (0.2 < 0.25) vẫn bị huỷ
go(g, along(0.2), probs=paper, shape=0, n=6); assert g.state == "NEED_A"
go.t = 0.0; g = GameAB(); start_go(g)                                   # chạm vạch (0.3) là đổi tay được ngay
go(g, along(0.3), probs=paper, shape=0); assert g.state == "GO" and g.passed and not g.warn
go(g, along(0.1), probs=paper, shape=0, n=10); assert g.state == "GO"   # đã qua vạch thì quay lại đổi tay cũng không sao (đã mở khoá)
print("Tất cả phép thử đều đạt (búa ở A + vạch).")

Tất cả phép thử đều đạt (búa ở A + vạch).


In [7]:
#cell 7: giao diện (chữ đen trên nền màu, chữ không dấu vì OpenCV không vẽ được dấu); vẽ ô A, ô B, đường nối, vòng giữ búa, vạch
COLORS = {0: (60, 60, 255), 1: (255, 150, 40), 2: (60, 210, 60)}      # BGR: paper đỏ, rock xanh dương, scissors xanh lá
GRAY, YELLOW, BLACK, WHITE = (190, 190, 190), (150, 255, 255), (0, 0, 0), (255, 255, 255)
RED, CYAN, ORANGE, GREEN = (40, 40, 255), (255, 230, 0), (60, 160, 255), (80, 230, 80)
BONES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
         (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]
BANNER = {"ai": ("AI THANG", (60, 60, 255)), "player": ("BAN THANG", (60, 210, 60)),
          "draw": ("HOA", YELLOW), "unknown": ("KHONG XAC DINH", GRAY)}

def put_label(img, text, x, y, bg, scale=0.5):
    th = 1 if scale < 0.7 else 2                                  # chữ nhỏ thì nét mảnh cho dễ đọc
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, th)
    cv2.rectangle(img, (x, y - h - base - 4), (x + w + 8, y + 2), bg, -1)
    cv2.putText(img, text, (x + 4, y - base), cv2.FONT_HERSHEY_SIMPLEX, scale, BLACK, th, cv2.LINE_AA)

def draw_hand(img, lm, color, mirror=True):
    h, w = img.shape[:2]
    px, py = (lm[:, 0] * w).astype(int), (lm[:, 1] * h).astype(int)
    if mirror: px = w - 1 - px
    for a, b in BONES:
        cv2.line(img, (px[a], py[a]), (px[b], py[b]), color, 2, cv2.LINE_AA)
    cv2.rectangle(img, (px.min() - 12, py.min() - 12), (px.max() + 12, py.max() + 12), color, 2)

def draw_zone(img, g):
    """Vẽ dải cho phép, đường nối, ô A, ô B và vạch. Chờ ở A: ô A vàng, đang giữ búa thì xanh lá + vòng tiến độ;
    đi: đỏ khi lệch đường; vạch cam (chưa chạm, phải giữ búa) -> xanh lá (đã chạm, được đổi tay)."""
    h, w = img.shape[:2]
    a, b = (int(g.A_n[0] * w), int(g.A_n[1] * h)), (int(g.B_n[0] * w), int(g.B_n[1] * h))
    r, need = int(g.box_r * h), g.state == "NEED_A"
    col = RED if g.off else CYAN
    col_a = (GREEN if g.a_hold > 0 else YELLOW) if need else col
    col_b = GRAY if need else col
    col_line = GRAY if need else col
    overlay = img.copy()
    cv2.line(overlay, a, b, col_line, max(2, int(2 * g.band * h)), cv2.LINE_AA)       # dải cho phép lệch (mờ)
    cv2.addWeighted(overlay, 0.25, img, 0.75, 0, img)
    cv2.line(img, a, b, col_line, 2, cv2.LINE_AA)
    for pt, name, c in ((a, "A", col_a), (b, "B", col_b)):
        cv2.circle(img, pt, r, c, 3, cv2.LINE_AA)
        put_label(img, name, pt[0] - 8, pt[1] - r - 6, c, 0.5)
    put_label(img, "BUA", a[0] - 18, a[1] + r + 22, col_a, 0.5)                       # nhắc: ô A phải là búa
    if need and g.a_hold > 0:                                                         # vòng tiến độ giữ búa
        cv2.ellipse(img, a, (r + 8, r + 8), -90, 0, int(360 * g.a_hold), GREEN, 4, cv2.LINE_AA)
    if g.state in ("NEED_A", "GO"):                                                   # vạch vuông góc với đường A->B tại gate_f
        d = np.array([b[0] - a[0], b[1] - a[1]], np.float32); d /= np.linalg.norm(d)
        perp = np.array([-d[1], d[0]])
        c = np.array(a, np.float32) + (np.array(b, np.float32) - np.array(a, np.float32)) * g.gate_f
        half = 1.3 * g.band * h
        gc = GREEN if (g.state == "GO" and g.passed) else (YELLOW if need else ORANGE)
        cv2.line(img, tuple((c - perp * half).astype(int)), tuple((c + perp * half).astype(int)), gc, 4, cv2.LINE_AA)

def draw_game_ab(img, g):
    h, w = img.shape[:2]
    if g.state == "GO":
        if g.off: txt, c = "LECH DUONG!", RED
        elif g.warn: txt, c = "CANH BAO: GIU BUA toi vach!", RED
        elif not g.passed: txt, c = "Giu BUA toi vach", YELLOW
        else: txt, c = "Di theo duong toi B", YELLOW
        put_label(img, txt, 10, h - 32, c, 0.55)
    elif g.state == "RESULT":
        text, col = BANNER[g.outcome]
        put_label(img, text, 10, 100, col, 1.1)
        if g.outcome == "unknown":
            put_label(img, "Khong doc duoc hinh tay sau khi AI chot", 10, h - 56, GRAY, 0.5)
        else:
            put_label(img, f"BAN ra: {CLASSES[g.player]}", 10, h - 80, COLORS[g.player], 0.6)
            put_label(img, f"AI doan ban ra {CLASSES[g.guess]} -> AI ra: {CLASSES[g.ai]}", 10, h - 56, COLORS[g.ai], 0.6)
        put_label(img, "Khung luc cham B | SPACE: choi tiep", 10, h - 32, YELLOW, 0.5)
    elif g.state == "CANCEL":
        put_label(img, "VAN HUY", 10, 100, ORANGE, 1.1)
        put_label(img, f"Tay luc cham B khac tay sau {g.confirm_sec:.1f}s", 10, h - 56, ORANGE, 0.55)
        put_label(img, "Khung luc cham B | SPACE: choi lai", 10, h - 32, YELLOW, 0.5)
    elif g.state != "CONFIRM":
        put_label(img, g.msg, 10, h - 32, YELLOW, 0.55)
    s = g.score
    put_label(img, f"AI {s['ai']} - {s['player']} Nguoi | hoa {s['draw']} | huy {g.stats['cancel']}", 10, h - 10, YELLOW, 0.45)
print("Đã định nghĩa giao diện.")

Đã định nghĩa giao diện.


In [8]:
#cell 8: âm thanh: tick cho nhịp 3-2-1, "ra" cao và dài hơn; file wav tự tạo trong Data_321\sounds (có rồi thì không tạo lại)
import wave, winsound
SND = BASE / "sounds"; SND.mkdir(exist_ok=True)

def make_wav(path, freq, ms, vol=0.6):
    sr = 44100; t = np.arange(int(sr * ms / 1000)) / sr
    x = np.sin(2 * np.pi * freq * t) * np.minimum(1, np.minimum(t, t[-1] - t) * 200)   # mờ đầu/cuối, tránh tiếng "tạch"
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes((x * vol * 32767).astype(np.int16).tobytes())

SOUNDS = {"tick": (SND / "tick.wav", 800, 120), "ra": (SND / "ra.wav", 1400, 350)}
for path, f, ms in SOUNDS.values():
    if not path.exists(): make_wav(path, f, ms)

def play(name):                                       # phát không chặn vòng lặp webcam
    winsound.PlaySound(str(SOUNDS[name][0]), winsound.SND_FILENAME | winsound.SND_ASYNC)
print("Âm thanh ở:", SND)

Âm thanh ở: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\sounds


In [9]:
#cell 9: chạy webcam + game A -> B — SPACE: chơi / chơi tiếp / chơi lại | r: reset điểm | q: thoát
CAM_ID, MIRROR, HAND_MODE, SHOW_DEBUG = 0, True, "palm", True      # SHOW_DEBUG=True hiện nhãn model trên hình (chỉ để test, game thật phải tắt)
cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480); cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
assert cap.isOpened(), "Không mở được webcam (thử đổi CAM_ID sang 1)."

buf.clear(); state.update(prev_lm=None, prev_t=None)
g = GameAB(A=(0.2, 0.5), B=(0.8, 0.5), box_r=0.13, band=0.09, confirm_sec=0.2)     # A trái, B phải; đổi tham số ở đây
print("A (trai):", g.A_n, "| B (phai):", g.B_n, "| ban kinh o:", g.box_r)          # kiểm tra nhanh: phải ra A (0.2, 0.5) và B (0.8, 0.5)
evidence, fps, last, shown = None, 0.0, time.perf_counter(), 0
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        now = time.perf_counter()
        lm = detect(frame, int((now - T0) * 1000))
        push(lm, now)
        pr = probs_now() if lm is not None else None
        sh = int(hand_shape(lm)) if lm is not None else None
        pt = hand_point(lm, MIRROR, HAND_MODE) if lm is not None else None
        g.step(now, pt, pr, sh)                   # luật chạy trước khi vẽ nên đáp án AI hiện ngay ở khung chốt
        if g.entered_go: play("tick")             # tiếng tick khi tay chạm A
        if g.locked_now: play("ra")               # tiếng cao khi tay chạm B (AI chốt)
        if len(g.history) > shown:
            shown = len(g.history); print(f"Van {shown}:", g.history[-1])

        disp = cv2.flip(frame, 1) if MIRROR else frame
        draw_zone(disp, g)
        if pr is not None:
            c, conf = int(pr.argmax()), float(pr.max())
            col = COLORS[c] if conf >= g.thr else GRAY
            draw_hand(disp, lm, col, MIRROR)
            hh, ww = disp.shape[:2]
            cv2.circle(disp, (int(pt[0] * ww), int(pt[1] * hh)), 6, WHITE, -1)     # điểm đại diện của tay
            if SHOW_DEBUG:
                put_label(disp, f"model: {CLASSES[c]}{'' if conf >= g.thr else '?'} {conf:.0%} | hinh: {CLASSES[sh] if sh >= 0 else '?'}", 10, 42, col, 0.45)
        if g.locked_now:
            evidence = disp.copy()                # khung lúc chạm B (có đường, ô, khung xương)
        if g.state in ("RESULT", "CANCEL") and evidence is not None:
            disp = evidence.copy()                # kết quả hoặc huỷ ván đứng yên trên khung lúc chạm B cho đến khi bấm SPACE
        fps, last = 0.9 * fps + 0.1 / max(now - last, 1e-6), now
        put_label(disp, f"FPS {fps:.0f}", 10, 20, YELLOW, 0.45)
        draw_game_ab(disp, g)
        cv2.imshow("RPS game A->B (SPACE: choi | r: reset | q: thoat)", disp)

        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"): break
        if k == 32: g.start(now)
        if k == ord("r"): g.score = {"ai": 0, "player": 0, "draw": 0}
finally:
    cap.release(); cv2.destroyAllWindows()
print("Đã đóng webcam. Số ván:", g.stats, "| điểm:", g.score)


A (trai): (0.2, 0.5) | B (phai): (0.8, 0.5) | ban kinh o: 0.13
Van 1: {'tu_A_den_chot_s': 8.86, 'model_doan': 0, 'tay_sau_xac_nhan': 0, 'khung_xac_nhan': [0, 0, 0, 0, 0, 0], 'ket_qua': 'RESULT', 'outcome': 'ai'}
Van 2: {'tu_A_den_chot_s': 1.04, 'model_doan': 2, 'tay_sau_xac_nhan': 2, 'khung_xac_nhan': [2, 2, 2, 2, 2, 2, 2], 'ket_qua': 'RESULT', 'outcome': 'ai'}
Van 3: {'tu_A_den_chot_s': 1.33, 'model_doan': 2, 'tay_sau_xac_nhan': 2, 'khung_xac_nhan': [2, 2, 2, 2, 2, 2], 'ket_qua': 'RESULT', 'outcome': 'ai'}
Van 4: {'tu_A_den_chot_s': 1.73, 'model_doan': 2, 'tay_sau_xac_nhan': 2, 'khung_xac_nhan': [2, 2, 2, 2, 2, 2, 2], 'ket_qua': 'RESULT', 'outcome': 'ai'}
Van 5: {'tu_A_den_chot_s': 0.86, 'model_doan': 0, 'tay_sau_xac_nhan': 0, 'khung_xac_nhan': [0, 0, 0, 0, 0, 0], 'ket_qua': 'RESULT', 'outcome': 'ai'}
Van 6: {'tu_A_den_chot_s': 0.59, 'model_doan': 0, 'tay_sau_xac_nhan': 0, 'khung_xac_nhan': [0, 0, 0, 0, 0, 0], 'ket_qua': 'RESULT', 'outcome': 'ai'}
Đã đóng webcam. Số ván: {'ok': 6, 'ca